In [4]:
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
import pandas as pd

In [6]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
from src.utils import *
set_plot_style()
print("setup ok")

setup ok


In [7]:
raw = pd.read_csv(DATA_RAW, parse_dates=["Date"])
df = raw[(raw["City"] == CITY) & raw["Date"].between(WINDOW_START, WINDOW_END)].copy()
df = df.dropna(subset=POLLUTANTS + ["AQI"])
df["Month"] = df["Date"].dt.month
df["Season"] = df["Month"].map(season_of)
df["Poor_Flag"] = df["AQI"].map(poor_flag)
print(df.shape)
print(df["Poor_Flag"].value_counts(normalize=True).round(3))

(1698, 19)
Poor_Flag
1    0.723
0    0.277
Name: proportion, dtype: float64


In [8]:
season_d = pd.get_dummies(df["Season"]).drop(columns="Winter").astype(float)
X = pd.concat([df[POLLUTANTS], season_d], axis=1)
y = df["Poor_Flag"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE,
                                          random_state=SEED, stratify=y)

logit = sm.Logit(y_tr, sm.add_constant(X_tr)).fit(disp=0)
print(logit.summary())

                           Logit Regression Results                           
Dep. Variable:              Poor_Flag   No. Observations:                 1358
Model:                          Logit   Df Residuals:                     1345
Method:                           MLE   Df Model:                           12
Date:                Mon, 12 Oct 2026   Pseudo R-squ.:                  0.7418
Time:                        10:05:31   Log-Likelihood:                -207.13
converged:                       True   LL-Null:                       -802.15
Covariance Type:            nonrobust   LLR p-value:                2.408e-247
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const          -11.7529      1.198     -9.812      0.000     -14.101      -9.405
PM2.5            0.0526      0.010      5.386      0.000       0.033       0.072
PM10             0.0280      0.003      8.46